In [ ]:
# DS528 Final Project
# Predicting Fan Travel Demand for the 2026 FIFA World Cup
# Binary classification + ROI-based model evaluation

In [ ]:
# Import required libraries

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)

In [ ]:
# Load synthetic dataset
# Each row represents a potential World Cup fan

data_path = Path("../data/synthetic_worldcup_fans.csv")
df = pd.read_csv(data_path)

df.head()

In [ ]:
# Check dataset shape and target distribution

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df["will_travel"].value_counts(normalize=True).rename("target_ratio").to_frame()

In [ ]:
# Define target variable
# will_travel = 1 means the fan is likely to travel / interested
# will_travel = 0 means the fan is unlikely to travel / low interest

target = "will_travel"

# Drop columns that should not be used as model features
# fan_id is only an identifier
# travel_probability_synthetic was used to generate the synthetic target
# expected_value_usd is derived from synthetic probability, so it may cause leakage

drop_cols = [
    "fan_id",
    "will_travel",
    "travel_probability_synthetic",
    "expected_value_usd"
]

X = df.drop(columns=drop_cols)
y = df[target]

In [ ]:
# Define categorical and numerical features

categorical_features = [
    "country_region",
    "income_level",
    "match_importance"
]

numeric_features = [
    col for col in X.columns
    if col not in categorical_features
]

print("Categorical features:", categorical_features)
print("Number of numeric features:", len(numeric_features))

In [ ]:
# Split data into train and test sets
# Stratify keeps the same target distribution in train and test data

X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
    X,
    y,
    df,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])

In [ ]:
# Preprocessing for Logistic Regression
# Numeric features are scaled
# Categorical features are one-hot encoded

preprocess_for_lr = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# Preprocessing for tree-based models
# Scaling is not required for tree-based models

preprocess_for_tree = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [ ]:
# Define machine learning models

models = {
    "Logistic Regression": Pipeline(steps=[
        ("preprocess", preprocess_for_lr),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline(steps=[
        ("preprocess", preprocess_for_tree),
        ("model", RandomForestClassifier(
            n_estimators=250,
            max_depth=10,
            min_samples_leaf=30,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]),

    "Gradient Boosting": Pipeline(steps=[
        ("preprocess", preprocess_for_tree),
        ("model", GradientBoostingClassifier(
            n_estimators=160,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        ))
    ])
}

In [ ]:
# Business impact function
# The model decides whether to send a targeted campaign
#
# TP: interested fan correctly targeted
# FP / Type-1 error: uninterested fan targeted, campaign cost is wasted
# FN / Type-2 error: interested fan missed, potential revenue is lost
# TN: uninterested fan not targeted, no direct cost

def calculate_business_impact(y_true, y_pred, test_frame):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    revenue = test_frame["potential_net_revenue_usd"].values
    cost = test_frame["campaign_cost_usd"].values

    tp_mask = (y_true == 1) & (y_pred == 1)
    fp_mask = (y_true == 0) & (y_pred == 1)
    fn_mask = (y_true == 1) & (y_pred == 0)
    tn_mask = (y_true == 0) & (y_pred == 0)

    tp_impact = np.sum(revenue[tp_mask] - cost[tp_mask])
    fp_impact = -np.sum(cost[fp_mask])
    fn_impact = -np.sum(revenue[fn_mask])

    net_impact = tp_impact + fp_impact + fn_impact

    return {
        "TP": int(tp_mask.sum()),
        "FP": int(fp_mask.sum()),
        "FN": int(fn_mask.sum()),
        "TN": int(tn_mask.sum()),
        "TP_impact_usd": round(tp_impact, 2),
        "FP_impact_usd": round(fp_impact, 2),
        "FN_impact_usd": round(fn_impact, 2),
        "Net_business_impact_usd": round(net_impact, 2)
    }

In [ ]:
# Train models and evaluate both technical performance and business impact

results = []
predictions = {}

for model_name, model in models.items():
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    impact = calculate_business_impact(y_test, y_pred, df_test)

    results.append({
        "Model": model_name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall": round(recall_score(y_test, y_pred), 4),
        "F1": round(f1_score(y_test, y_pred), 4),
        "ROC_AUC": round(roc_auc_score(y_test, y_proba), 4),
        **impact
    })

    predictions[model_name] = {
        "model": model,
        "y_pred": y_pred,
        "y_proba": y_proba
    }

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("Net_business_impact_usd", ascending=False)

results_df

In [ ]:
# Save model comparison output

output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)

results_df.to_csv(output_dir / "model_comparison_with_roi.csv", index=False)

In [ ]:
# Threshold optimization
# Default threshold is 0.50, but this may not maximize ROI
# Because False Negatives are costly, lower thresholds may perform better financially

thresholds = np.round(np.arange(0.10, 0.91, 0.05), 2)
threshold_results = []

for model_name, pred_data in predictions.items():
    y_proba = pred_data["y_proba"]

    for threshold in thresholds:
        y_pred_threshold = (y_proba >= threshold).astype(int)
        impact = calculate_business_impact(y_test, y_pred_threshold, df_test)

        threshold_results.append({
            "Model": model_name,
            "Threshold": threshold,
            "Accuracy": round(accuracy_score(y_test, y_pred_threshold), 4),
            "Precision": round(precision_score(y_test, y_pred_threshold, zero_division=0), 4),
            "Recall": round(recall_score(y_test, y_pred_threshold, zero_division=0), 4),
            "F1": round(f1_score(y_test, y_pred_threshold, zero_division=0), 4),
            **impact
        })

threshold_df = pd.DataFrame(threshold_results)

best_thresholds_df = (
    threshold_df
    .sort_values(["Model", "Net_business_impact_usd"], ascending=[True, False])
    .groupby("Model")
    .head(1)
    .sort_values("Net_business_impact_usd", ascending=False)
)

best_thresholds_df

In [ ]:
# Save threshold optimization outputs

threshold_df.to_csv(output_dir / "roi_by_threshold.csv", index=False)
best_thresholds_df.to_csv(output_dir / "best_thresholds_by_model.csv", index=False)

In [ ]:
# Plot net business impact by threshold

plt.figure(figsize=(10, 6))

for model_name in threshold_df["Model"].unique():
    temp = threshold_df[threshold_df["Model"] == model_name]
    plt.plot(
        temp["Threshold"],
        temp["Net_business_impact_usd"],
        marker="o",
        label=model_name
    )

plt.title("Net Business Impact by Classification Threshold")
plt.xlabel("Classification Threshold")
plt.ylabel("Net Business Impact (USD)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(output_dir / "roi_by_threshold_plot.png", dpi=200)
plt.show()

In [ ]:
# Feature importance for Gradient Boosting

def get_feature_names_from_pipeline(pipeline, categorical_features, numeric_features):
    preprocessor = pipeline.named_steps["preprocess"]
    cat_encoder = preprocessor.named_transformers_["cat"]
    cat_names = cat_encoder.get_feature_names_out(categorical_features)
    return np.concatenate([numeric_features, cat_names])

gb_pipeline = predictions["Gradient Boosting"]["model"]

feature_names = get_feature_names_from_pipeline(
    gb_pipeline,
    categorical_features,
    numeric_features
)

importances = gb_pipeline.named_steps["model"].feature_importances_

feature_importance_df = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    })
    .sort_values("importance", ascending=False)
)

feature_importance_df.head(10)

In [ ]:
# Save feature importance output

feature_importance_df.to_csv(
    output_dir / "gradient_boosting_feature_importance.csv",
    index=False
)

In [ ]:
# Plot top feature importances

top_features = feature_importance_df.head(12).sort_values("importance", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(top_features["feature"], top_features["importance"])
plt.title("Top Feature Importance - Gradient Boosting")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()

plt.savefig(output_dir / "gradient_boosting_feature_importance.png", dpi=200)
plt.show()

In [ ]:
# Final business conclusion
#
# The best model should not be selected only by accuracy.
# In this project, False Negatives are expensive because they represent missed revenue.
# Therefore, the classification threshold should be selected based on Net Business Impact.
#
# Main output tables:
# 1. results_df
# 2. best_thresholds_df
# 3. feature_importance_df

print("Best model by default threshold ROI:")
display(results_df.head(1))

print("Best model-threshold combination by ROI:")
display(best_thresholds_df.head(1))